[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/12_mcp_tools/12_proyecto_mcp_tools.ipynb)

# Proyecto 12 · Instrumentar el servidor MCP de pedidos y unir su traza con la del agente

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno (todo se evalúa en memoria; lanza un proceso hijo por stdio). Opcional: `core` + `metrics` para el paso final |
| **RAM aproximada** | < 700 MB |
| **Prerrequisitos** | Lección `12_mcp_tools` y módulo 11 (propagación de contexto, resiliencia) |
| **Conexión** | Sustituye las herramientas locales de SupportOps por un servidor MCP **observable y seguro**; el módulo 13 usará estas trazas para atribuir el coste |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. El equipo de pedidos ha expuesto sus herramientas como un **servidor MCP** y el agente ya las usa. Tras una semana en producción hay quejas:

1. **Hay trazas del servidor que nadie sabe a qué ticket pertenecen** (el *dashboard* de trazas partidas se llena cuando el agente paraleliza herramientas).
2. **Un reembolso se pagó dos veces** tras un *timeout*.
3. **Seguridad pide** que la telemetría no guarde datos personales, que el servidor no obedezca contextos ajenos y que se **detecte** si alguien cambia las descripciones de las herramientas.
4. **Observabilidad pide** un contrato: que los atributos de la spec MCP que usan los paneles estén de verdad en los spans.

## Datos

Todo sintético y determinista (semilla 42, `mock`). El **Paso 0** lanza el servidor MCP de pedidos como proceso hijo y ejecuta 10 tickets reales del agente más 6 llamadas desde hilos sin contexto; te entrega `AGENT_ROWS` y `SERVER_ROWS` (spans de los dos procesos, esquema `SPAN_SCHEMA`). Cadena de datos: stack local (opcional) → memoria + JSONL.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `safe_args` | 8 vectores: lista blanca por herramienta, PII enmascarada en valores permitidos, resto `<redacted>` | 8/8 |
| R2 | `ToolPolicyMiddleware` | Sobre un servidor en proceso: atributos de herramienta, evento de **auditoría** solo en destructivas, histograma `mcp.server.operation.duration` por herramienta/error, clase de error `caller`/`server`, sin argumentos capturados | 6/6 comprobaciones |
| R3 | `join_report` | Llamadas de servidor, **unidas** al agente, **huérfanas** y mediana de *overhead* cliente−servidor | contadores exactos · overhead entre 0 y 100 ms |
| R4 | `safe_call` | 4 escenarios: OK (1 intento), lectura lenta (2 intentos, `timeout`), `tool_error` (1 intento, sin reintento), reembolso con *timeout* (2 intentos, **1** ejecución real) | 4/4 |
| R5 | `tools_digest`, `detect_drift` | Estable ante reordenación, sensible a cualquier cambio de descripción; detecta el servidor alterado | 5/5 |
| R6 | `spec_gaps` | Sobre 3 spans sintéticos: lista exacta de atributos de la spec que faltan por nivel; y el span servidor real **no** pierde ninguno requerido | exacto |

> 📏 Los umbrales salen de la solución de referencia (semilla 42, fija en el notebook; con otra semilla cambian los tickets y, por tanto, los contadores del R3). Las latencias son de pared (compresión ×50), por eso solo se exigen **rangos amplios** y relaciones (cliente ≥ servidor), no valores absolutos.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
# El SDK oficial de Python de MCP (v2.x). La versión 1.x tenía otra API (FastMCP); este módulo usa `mcp.server.mcpserver.MCPServer` y `mcp.Client`.
%pip install -q "mcp>=2,<3"

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import asyncio, contextvars, dataclasses, inspect, json, time
from concurrent.futures import ThreadPoolExecutor
from opentelemetry import trace as otel_trace
from opentelemetry.trace import SpanKind, Status, StatusCode
from supportops import catalog
from supportops.instrument import CompositeInstrumentor
from supportops.telemetry import setup_telemetry
from supportops.tools import ToolError
from supportops.mcp_orders import (TOOL_META, McpServerProcess, McpToolBridge, McpToolSpanEnricher, WallClockInstrumentor, build_server, redact_args,
                                   tools_fingerprint, tracking_rule)
from supportops.multiagent import agent_tree_text, attached, extract_manual, inject_manual, parse_traceparent, service_palette, waterfall_services

os.environ["SUPPORTOPS_REAL_SLEEP_SCALE"] = "0.02"      # 1 s simulado del agente = 20 ms de pared (los spans del SDK de MCP usan el reloj REAL)
STACK = check_stack("core")
LIVE = not STACK.degraded                                  # True: el Collector responde y enviamos la telemetría de verdad
TEL = setup_telemetry("supportops-agent", mode="memory")        # en memoria: la rúbrica no depende del stack

def run_async(coro):
    """Ejecuta una corrutina desde una celda de Jupyter (que ya tiene un bucle) usando un hilo con su propio bucle."""
    with ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, coro).result()

import hashlib, re
from mcp import Client
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from supportops.mcp_orders import MCP_BUCKETS

## Paso 0 · El sistema de partida (te lo damos hecho)

Lanza el servidor MCP de pedidos como proceso hijo (con **nuestra** telemetría de servidor ya activa para que puedas inspeccionarla), conecta el puente, ejecuta 10 tickets del agente y 6 llamadas **desde hilos sin copiar el contexto** (el origen del problema 1).

In [ ]:
SP = McpServerProcess(time_scale=0.02)
BR = McpToolBridge(SP.params(), read_timeout_s=3.0, meter=TEL.meter())
registry = default_tools(); remote = BR.register_tools(registry, ["get_order", "track_shipment", "issue_refund"])
llm = MockLLM("mock-small"); llm.add_rule(tracking_rule())
agent = SupportAgent(llm=llm, tools=registry, instrumentor=CompositeInstrumentor(WallClockInstrumentor(TEL), McpToolSpanEnricher(remote)))
pool = [t for t in generate_tickets(120, seed=SEED) if t.intent_true in ("order_status", "refund")][:10]
for t in pool: agent.run(dataclasses.replace(t, faults=[], created_at=time.time()))
with TEL.tracer().start_as_current_span("job-batch (hilos sin contexto)") as _sp, ThreadPoolExecutor(2) as ex:
    [f.result() for f in [ex.submit(BR.call, "get_order", {"order_id": f"ORD-{300 + i:05d}"}) for i in range(6)]]
time.sleep(0.8)
AGENT_ROWS, SERVER_ROWS = TEL.span_records(), SP.spans()
print(f"agente: {len(AGENT_ROWS)} spans · servidor MCP: {len(SERVER_ROWS)} spans · tools/call en el servidor: {sum(r['name'].startswith('tools/call') for r in SERVER_ROWS)}")
display(pd.DataFrame([{"span": r["name"], "servicio": r["resource"]["service.name"].replace("supportops-", ""), "kind": r["kind"]} for r in (AGENT_ROWS[:3] + SERVER_ROWS[:3])]))

## Paso 1 · Telemetría que no filtra datos personales

**TODO 1.** `safe_args(tool, args)` devuelve los argumentos **aptos para telemetría**:

- Solo las claves de la **lista blanca** de la herramienta (`ALLOW`) conservan su valor; **cualquier otra** clave pasa a `"<redacted>"` (aunque sea un número).
- En los valores **de texto permitidos** hay que enmascarar PII con los patrones `PII` (correo → `<email>`, tarjeta de 13-19 dígitos → `<tarjeta>`).
- Herramientas sin entrada en `ALLOW`: todo `<redacted>`. No debe mutar el diccionario de entrada.

<details>
<summary>🪜 Pista 1</summary>

Recorre `args.items()`: si la clave no está en `ALLOW.get(tool, set())` → `"<redacted>"`. Si el valor es `str`, aplica cada `(patrón, sustituto)` de `PII` con `patrón.sub(sustituto, valor)`.

</details>

In [ ]:
ALLOW = {"get_order": {"order_id"}, "list_customer_orders": {"limit"}, "check_stock": {"sku"}, "track_shipment": {"order_id"}, "issue_refund": {"order_id", "amount_eur"}}
PII = [(re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"), "<email>"), (re.compile(r"\b(?:\d[ -]?){13,19}\b"), "<tarjeta>")]

def safe_args(tool: str, args: dict) -> dict:
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    print(safe_args("issue_refund", {"order_id": "ORD-00002 ana@example.com", "amount_eur": 9.5, "reason": "tarjeta 4111 1111 1111 1111"}))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Middleware de política y auditoría del servidor

El servidor se construye con `build_server("normal", telemetry=False, meter=...)`: solo trae la instrumentación del SDK. **TODO 2.** Completa `ToolPolicyMiddleware.__call__(ctx, call_next)` para `tools/call` (el resto de mensajes pasan intactos):

1. Atributos en el **span actual** (`otel_trace.get_current_span()`): `supportops.tool.read_only` y `supportops.tool.destructive` (bool, de `TOOL_META`; si la herramienta no está en la tabla, ambos `False`) y `network.transport="pipe"`.
2. Tras llamar a `call_next`: el resultado llega al middleware en **forma de cable**, un `dict` (`{"content": [{"text": ...}], "isError": true|ausente, ...}`). Si es de error (`result.get("isError")`), el tipo de error es `"tool_error"` **y** pon el atributo `supportops.error.class`: `"caller"` si el texto del resultado (unión de los `text` de `content`) empieza por `Unknown tool` o contiene `validation error`, si no `"server"`.
3. **Herramientas destructivas** (`TOOL_META[tool]["destructive"]`): añade **siempre** (también si falla) un evento de span `supportops.audit` con `{"tool", "outcome": "ok"|"error", "order_id": <primeros 16 caracteres del argumento order_id o "">}`.
4. **Métrica** `mcp.server.operation.duration` (`self.hist.record(segundos, atributos)`) con `mcp.method.name="tools/call"`, `gen_ai.tool.name`, `gen_ai.operation.name="execute_tool"` y `error.type` **solo si hubo error**.
5. **No** captures argumentos como atributos.

<details>
<summary>🪜 Pista 1</summary>

Estructura: lee `name = ctx.params['name']`, `args = ctx.params.get('arguments') or {}`; marca `t0 = time.perf_counter()`; `err = None`; `try: result = await call_next(ctx) ... finally:` registra evento y métrica. Recuerda `return result` y que una excepción debe **relanzarse**.

</details>

<details>
<summary>🪜 Pista 2</summary>

Texto del resultado: `"".join(c.get('text', '') for c in result['content'])`. El evento: `span.add_event('supportops.audit', {...})` dentro del `finally` para que se registre aunque haya fallo.

</details>

In [ ]:
class ToolPolicyMiddleware:
    def __init__(self, meter):
        self.hist = meter.create_histogram("mcp.server.operation.duration", unit="s", explicit_bucket_boundaries_advisory=MCP_BUCKETS)

    async def __call__(self, ctx, call_next):
        # TODO 2
        raise NotImplementedError("TODO 2")

try:
    from opentelemetry.sdk.metrics import MeterProvider
    from opentelemetry.sdk.metrics.export import InMemoryMetricReader
    _r = InMemoryMetricReader(); _srv = build_server("normal", telemetry=False); _srv.middleware.append(ToolPolicyMiddleware(MeterProvider(metric_readers=[_r]).get_meter("t")))
    async def _go():
        async with Client(_srv) as c: return (await c.call_tool("get_order", {"order_id": "ORD-00001"})).is_error
    print("llamada de prueba, is_error =", run_async(_go()))
except NotImplementedError as e:
    print("Pendiente:", e)
except Exception as e:
    print("Pendiente (aún falla):", type(e).__name__, str(e)[:80])

## Paso 3 · Unir las trazas: informe de cobertura

**TODO 3.** `join_report(agent_rows, server_rows)` devuelve un `dict`:

- `tool_calls`: nº de spans del servidor cuyo nombre empieza por `tools/call`.
- `joined`: cuántos de ellos pertenecen a una traza que **también contiene un span `invoke_agent`** del agente.
- `orphans`: `tool_calls − joined`.
- `overhead_ms_median`: mediana (ms, `float`) de `duración del span CLIENTE padre − duración del span servidor` sobre las llamadas **unidas** (el padre del span servidor es el span `MCP send tools/call …` de `agent_rows`). Si no hay ninguna unida, `None`.
- `by_tool`: `dict` con el nº de llamadas por `gen_ai.tool.name`.

<details>
<summary>🪜 Pista 1</summary>

Conjunto de trazas con agente: `{r['trace_id'] for r in agent_rows if r['name'].startswith('invoke_agent')}`. Índice de spans del agente por `span_id` para buscar el CLIENTE padre.

</details>

In [ ]:
def join_report(agent_rows: list[dict], server_rows: list[dict]) -> dict:
    # TODO 3
    raise NotImplementedError("TODO 3")

try:
    print(join_report(AGENT_ROWS, SERVER_ROWS))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Llamar con política: timeouts, reintentos e idempotencia

**TODO 4.** `safe_call(bridge, name, args, timeout_s, retries=1, backoff_s=0.05)` envuelve `bridge.call` y devuelve `{"ok", "value", "error_type", "attempts"}`:

- `ToolError` con `error_type` **`timeout`** o **`connection_closed`** → se **reintenta** hasta `retries` veces, esperando `backoff_s` antes de cada reintento, **solo si es seguro**: la herramienta es de **solo lectura** (`TOOL_META[name]["read_only"]`) **o** es destructiva y los `args` llevan `idempotency_key` (si falta, **añádela tú** de forma estable: los primeros 16 hex de `sha256` del JSON de `[name, args]` con claves ordenadas, **antes** del primer intento).
- `tool_error` y `protocol_error` **nunca** se reintentan.
- `ok=True` y `value` con el resultado si algún intento funciona; si no, `ok=False`, `value=None` y el `error_type` del último fallo.

<details>
<summary>🪜 Pista 1</summary>

Bucle `for attempt in range(1, retries + 2)`: `try: return {..., 'attempts': attempt}` / `except ToolError as e:` decide si reintentar (`e.error_type in ('timeout', 'connection_closed')` y `safe`) con `time.sleep(backoff_s)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Para las destructivas, calcula la clave **una vez antes del bucle**: `hashlib.sha256(json.dumps([name, args], sort_keys=True).encode()).hexdigest()[:16]` y haz `args = {**args, 'idempotency_key': key}` si no estaba.

</details>

In [ ]:
def safe_call(bridge, name: str, args: dict, timeout_s: float, retries: int = 1, backoff_s: float = 0.05) -> dict:
    # TODO 4
    raise NotImplementedError("TODO 4")

try:
    print(safe_call(BR, "get_order", {"order_id": "ORD-00001"}, 3.0))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 5 · Vigilar las herramientas: huella y deriva

**TODO 5.** (a) `tools_digest(tools)`: SHA-256 (primeros 16 hex) de las herramientas del servidor (`tools` = lista de objetos con `.name`, `.description` y `.input_schema`): debe ser **independiente del orden** y cambiar con **cualquier** cambio de nombre, descripción o esquema. (b) `detect_drift(approved, tools)` devuelve un `dict` `{"drifted": bool, "added": [...], "removed": [...], "changed": [...]}` comparando la lista **aprobada** (`dict` nombre → huella por herramienta, que genera `per_tool_digest`) con la actual; `changed` = herramientas que existen en ambas con huella distinta.

<details>
<summary>🪜 Pista 1</summary>

Serializa cada herramienta como `[name, description or '', input_schema or {}]` con `json.dumps(..., sort_keys=True)`, ordena la lista por nombre y haz el SHA-256 del conjunto. `per_tool_digest` es lo mismo pero por herramienta.

</details>

In [ ]:
def per_tool_digest(tools) -> dict:
    return {t.name: hashlib.sha256(json.dumps([t.name, t.description or "", getattr(t, "input_schema", None) or {}], sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16] for t in tools}

def tools_digest(tools) -> str:
    # TODO 5a
    raise NotImplementedError("TODO 5a")

def detect_drift(approved: dict, tools) -> dict:
    # TODO 5b
    raise NotImplementedError("TODO 5b")

try:
    print(tools_digest(BR.tools_list), detect_drift(per_tool_digest(BR.tools_list), BR.tools_list))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 6 · Un contrato de conformidad con la spec

**TODO 6.** `spec_gaps(span_row, kind)` devuelve `{"required": [...], "conditional": [...], "recommended": [...]}` con los atributos de `SPEC[kind]` (`kind` = `"client"` o `"server"`) que **faltan** en `span_row["attributes"]` (listas ordenadas alfabéticamente). Solo aplica a spans de `tools/call`. Úsala como **prueba de contrato** para un servidor real: `spec_gaps(<span servidor>, "server")["required"]` debe estar vacío.

In [ ]:
SPEC = {"client": {"required": ["mcp.method.name"], "conditional": ["gen_ai.tool.name", "jsonrpc.request.id"], "recommended": ["gen_ai.operation.name", "mcp.protocol.version", "network.transport"]},
        "server": {"required": ["mcp.method.name"], "conditional": ["gen_ai.tool.name"], "recommended": ["gen_ai.operation.name", "mcp.protocol.version", "network.transport"]}}

def spec_gaps(span_row: dict, kind: str) -> dict:
    # TODO 6
    raise NotImplementedError("TODO 6")

try:
    sv_ = next(r for r in SERVER_ROWS if r["name"].startswith("tools/call"))
    print(spec_gaps(sv_, "server"))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 7 · Rúbrica

In [ ]:
REF = {"tool_calls": 20, "joined": 14, "orphans": 6, "by_tool": {"get_order": 16, "issue_refund": 1, "track_shipment": 3}}     # calibrado con la solución (semilla 42)

def run_rubric():
    rows = []
    def add(r, d, ok, det=""): rows.append({"criterio": r, "descripción": d, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": det})
    # R1
    try:
        V = [(("get_order", {"order_id": "ORD-00001"}), {"order_id": "ORD-00001"}),
             (("get_order", {"order_id": "ORD-00001 ana@example.com"}), {"order_id": "ORD-00001 <email>"}),
             (("issue_refund", {"order_id": "ORD-00002", "amount_eur": 9.5, "reason": "x", "idempotency_key": "k"}), {"order_id": "ORD-00002", "amount_eur": 9.5, "reason": "<redacted>", "idempotency_key": "<redacted>"}),
             (("issue_refund", {"order_id": "pago 4111 1111 1111 1111"}), {"order_id": "pago <tarjeta>"}),
             (("check_stock", {"sku": "SKU-1001", "extra": 5}), {"sku": "SKU-1001", "extra": "<redacted>"}),
             (("list_customer_orders", {"customer_id": "C-0001", "limit": 3}), {"customer_id": "<redacted>", "limit": 3}),
             (("desconocida", {"x": 1, "y": "z"}), {"x": "<redacted>", "y": "<redacted>"}),
             (("get_order", {}), {})]
        orig = {"order_id": "ORD-00001 ana@example.com"}; keep = dict(orig); safe_args("get_order", orig)
        n = sum(safe_args(*a) == e for a, e in V) + (0 if orig != keep else 0)
        ok = n == 8 and orig == keep
        add("R1", "safe_args: 8 vectores y sin mutar la entrada", ok, f"{n}/8 · entrada intacta: {orig == keep}")
    except Exception as e: add("R1", "safe_args", False, f"{type(e).__name__}: {e}")
    # R2
    try:
        rd = InMemoryMetricReader(); mp = MeterProvider(metric_readers=[rd]); srv = build_server("normal", telemetry=False); srv.middleware.append(ToolPolicyMiddleware(mp.get_meter("r2")))
        TEL.reset_local()
        async def go():
            async with Client(srv) as c:
                await c.call_tool("get_order", {"order_id": "ORD-00001"})
                await c.call_tool("issue_refund", {"order_id": "ORD-00001", "amount_eur": 1.0, "idempotency_key": "r2"})
                await c.call_tool("no_existe", {})
        run_async(go()); time.sleep(0.2)
        sp = {r["name"]: r for r in TEL.span_records() if r["name"].startswith("tools/call")}
        g, rf, ne = sp.get("tools/call get_order"), sp.get("tools/call issue_refund"), sp.get("tools/call no_existe")
        ev = lambda r: [e for e in r["events"] if e["name"] == "supportops.audit"]
        pts = {(p.attributes["gen_ai.tool.name"], p.attributes.get("error.type")): p.count for rm in rd.get_metrics_data().resource_metrics for sm in rm.scope_metrics for m in sm.metrics
               if m.name == "mcp.server.operation.duration" for p in m.data.data_points}
        checks = {"atributos de la herramienta": bool(g) and g["attributes"].get("supportops.tool.read_only") is True and g["attributes"].get("supportops.tool.destructive") is False and g["attributes"].get("network.transport") == "pipe",
                  "auditoría solo en destructivas": bool(rf) and len(ev(rf)) == 1 and ev(rf)[0]["attributes"].get("tool") == "issue_refund" and ev(rf)[0]["attributes"].get("outcome") == "ok" and not ev(g),
                  "destructiva no marcada como solo lectura": bool(rf) and rf["attributes"].get("supportops.tool.destructive") is True and rf["attributes"].get("supportops.tool.read_only") is False,
                  "métrica por herramienta/error": pts == {("get_order", None): 1, ("issue_refund", None): 1, ("no_existe", "tool_error"): 1},
                  "clase de error caller": bool(ne) and ne["attributes"].get("supportops.error.class") == "caller" and ne["attributes"].get("error.type") == "tool_error",
                  "sin argumentos capturados": all("gen_ai.tool.call.arguments" not in r["attributes"] for r in sp.values())}
        add("R2", "ToolPolicyMiddleware: 6 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R2", "ToolPolicyMiddleware", False, f"{type(e).__name__}: {e}")
    # R3
    try:
        j = join_report(AGENT_ROWS, SERVER_ROWS)
        ok = (j["tool_calls"], j["joined"], j["orphans"]) == (REF["tool_calls"], REF["joined"], REF["orphans"]) and j["by_tool"] == REF["by_tool"] and j["overhead_ms_median"] is not None and 0 < j["overhead_ms_median"] < 100
        add("R3", "join_report: contadores exactos y overhead en (0, 100) ms", ok, f"{j} · ref {REF['tool_calls']}/{REF['joined']}/{REF['orphans']}")
    except Exception as e: add("R3", "join_report", False, f"{type(e).__name__}: {e}")
    # R4
    try:
        pre = BR.call("refund_stats")["applied"]
        sc = {}
        sc["ok"] = safe_call(BR, "get_order", {"order_id": "ORD-00001"}, 3.0)
        sc["lectura_lenta"] = safe_call(BR, "track_shipment", {"order_id": "ORD-00017"}, 0.1)
        sc["tool_error"] = safe_call(BR, "track_shipment", {"order_id": "ORD-00013"}, 3.0)
        o = next(x for x in catalog.orders().values() if x["status"] == "delivered" and (x["delivered_days_ago"] or 99) <= 30)
        sc["reembolso"] = safe_call(BR, "issue_refund", {"order_id": o["order_id"], "amount_eur": o["total_eur"], "reason": "r4"}, 0.001)
        applied = BR.call("refund_stats")["applied"] - pre
        good = {"ok": sc["ok"]["ok"] and sc["ok"]["attempts"] == 1,
                "lectura_lenta": (not sc["lectura_lenta"]["ok"]) and sc["lectura_lenta"]["attempts"] == 2 and sc["lectura_lenta"]["error_type"] == "timeout",
                "tool_error": (not sc["tool_error"]["ok"]) and sc["tool_error"]["attempts"] == 1 and sc["tool_error"]["error_type"] == "tool_error",
                "reembolso": sc["reembolso"]["attempts"] == 2 and applied == 1}
        add("R4", "safe_call: 4 escenarios (incluye 1 sola ejecución del reembolso)", all(good.values()), f"{good} · ejecuciones del reembolso: {applied}")
    except Exception as e: add("R4", "safe_call", False, f"{type(e).__name__}: {e}")
    # R5
    try:
        T = BR.tools_list; swapped = list(reversed(T))
        import copy
        T2 = copy.deepcopy(T); T2[0].description = (T2[0].description or "") + " "
        T3 = copy.deepcopy(T); T3[1].input_schema = {**(T3[1].input_schema or {}), "extra": 1}
        ap = per_tool_digest(T)
        d_same = detect_drift(ap, swapped); d_desc = detect_drift(ap, T2); d_new = detect_drift(ap, T[:-1])
        sp_bad = McpServerProcess(mode="poisoned")
        with McpToolBridge(sp_bad.params(), read_timeout_s=3.0) as bad: d_bad = detect_drift(ap, bad.tools_list)
        checks = {"estable al reordenar": tools_digest(T) == tools_digest(swapped), "cambia con la descripción": tools_digest(T) != tools_digest(T2), "cambia con el esquema": tools_digest(T) != tools_digest(T3),
                  "sin deriva = False": d_same == {"drifted": False, "added": [], "removed": [], "changed": []},
                  "detecta cambio/eliminación y el servidor alterado": d_desc["changed"] == [T[0].name] and d_new["removed"] == [T[-1].name] and d_bad["drifted"] and "get_order" in d_bad["changed"]}
        add("R5", "tools_digest y detect_drift", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R5", "huella de herramientas", False, f"{type(e).__name__}: {e}")
    # R6
    try:
        full = {"name": "tools/call x", "attributes": {"mcp.method.name": "tools/call", "gen_ai.tool.name": "x", "jsonrpc.request.id": "1", "gen_ai.operation.name": "execute_tool", "mcp.protocol.version": "v", "network.transport": "pipe"}}
        bare = {"name": "tools/call x", "attributes": {"mcp.method.name": "tools/call"}}
        part = {"name": "tools/call x", "attributes": {"mcp.method.name": "tools/call", "gen_ai.tool.name": "x", "mcp.protocol.version": "v"}}
        e1, e2, e3 = spec_gaps(full, "client"), spec_gaps(bare, "client"), spec_gaps(part, "server")
        real = spec_gaps(next(r for r in SERVER_ROWS if r["name"].startswith("tools/call")), "server")
        ok = (e1 == {"required": [], "conditional": [], "recommended": []} and e2 == {"required": [], "conditional": ["gen_ai.tool.name", "jsonrpc.request.id"], "recommended": ["gen_ai.operation.name", "mcp.protocol.version", "network.transport"]}
              and e3 == {"required": [], "conditional": [], "recommended": ["gen_ai.operation.name", "network.transport"]} and real["required"] == [])
        add("R6", "spec_gaps exacto y contrato del servidor real", ok, f"cliente vacío={e1 == {'required': [], 'conditional': [], 'recommended': []}} · real={real}")
    except Exception as e: add("R6", "spec_gaps", False, f"{type(e).__name__}: {e}")
    return pd.DataFrame(rows).set_index("criterio")

In [ ]:
# Ejecuta esta celda cuando hayas terminado los TODO
display(run_rubric())

---
## Paso final (opcional) · Míralo en Grafana

Con el stack arriba (`docker compose --profile core --profile metrics up -d`) cambia `TEL` a modo `otlp` y el servidor a `McpServerProcess(telemetry="otlp", otlp_endpoint="http://localhost:4318")`, repite el Paso 0 y abre el dashboard **«12 · MCP y herramientas remotas»**: el panel de **trazas partidas** debe mostrar las 6 llamadas desde hilos sin contexto; arregla el origen (usa `contextvars.copy_context().run`) y comprueba que baja a cero.

## Retos extra (nivel experto)

1. **Política `link`:** ejecuta el servidor con `context_policy="link"` y mide cuántas trazas separadas aparecen con tu `join_report`; ¿qué consulta TraceQL recupera «las llamadas enlazadas a una traza del agente»?
2. **Presupuesto de reintentos:** limita `safe_call` a un máximo del 10 % de reintentos sobre las últimas 50 llamadas y demuestra con el servidor en modo `slow` que evita la tormenta de reintentos.
3. **Cancelación real:** reescribe `track_shipment` como herramienta **asíncrona** (`async def`) con `anyio.sleep` y comprueba si la notificación de cancelación del SDK **sí** corta el trabajo (¿termina el span servidor antes? ¿qué cambia en el «zombi»?).
4. **Canal HTTP:** lanza el servidor con `--transport streamable-http` y conecta con `Client("http://127.0.0.1:8000/mcp")`; ¿qué spans HTTP adicionales aparecen y cómo los relacionas (enlace) con los spans MCP?
5. **Alerta de huella:** emite `supportops.mcp.tools_hash` como atributo del span de conexión y escribe la regla de alerta que dispare cuando cambie.

## Reflexión (producción e hilo conductor)

- ¿Qué parte de lo medido en el `join_report` pondrías en un SLO («% de llamadas MCP unidas a su traza de agente») y qué umbral tendría sentido?
- `safe_call` reintenta lecturas lentas. ¿Qué pasa con la carga del servidor si **todos** los agentes hacen lo mismo durante un incidente? (módulo 11: amplificación) ¿Dónde pondrías un disyuntor?
- El módulo 13 atribuirá **coste por herramienta**: las llamadas MCP no tienen tokens, pero sí **tiempo y dinero** (una pasarela de pagos cobra por transacción). ¿Qué atributos de span necesitarías para atribuirlo?
- Si el servidor MCP lo opera **otra empresa**, ¿qué parte de tu telemetría podrías seguir viendo y qué parte dependería de su colaboración (política `link`, `_meta`)?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine las funciones y vuelve a lanzar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def safe_args(tool, args):
    allowed = ALLOW.get(tool, set()); out = {}
    for k, v in args.items():
        if k not in allowed: out[k] = "<redacted>"; continue
        if isinstance(v, str):
            for pat, repl in PII: v = pat.sub(repl, v)
        out[k] = v
    return out

class ToolPolicyMiddleware:
    def __init__(self, meter):
        self.hist = meter.create_histogram("mcp.server.operation.duration", unit="s", explicit_bucket_boundaries_advisory=MCP_BUCKETS)

    async def __call__(self, ctx, call_next):
        if ctx.method != "tools/call":
            return await call_next(ctx)
        name = (ctx.params or {}).get("name"); args = (ctx.params or {}).get("arguments") or {}
        span = otel_trace.get_current_span(); meta = TOOL_META.get(name, {})
        span.set_attributes({"supportops.tool.read_only": bool(meta.get("read_only", False)), "supportops.tool.destructive": bool(meta.get("destructive", False)), "network.transport": "pipe"})
        t0 = time.perf_counter(); err = None
        try:
            result = await call_next(ctx)
            if (result or {}).get("isError") if isinstance(result, dict) else getattr(result, "is_error", False):
                err = "tool_error"
                text = "".join(c.get("text", "") for c in result.get("content", []))
                span.set_attribute("supportops.error.class", "caller" if (text.startswith("Unknown tool") or "validation error" in text) else "server")
            return result
        except Exception as e:
            err = type(e).__name__
            raise
        finally:
            if meta.get("destructive"):
                span.add_event("supportops.audit", {"tool": name, "outcome": "error" if err else "ok", "order_id": str(args.get("order_id", ""))[:16]})
            a = {"mcp.method.name": "tools/call", "gen_ai.tool.name": name, "gen_ai.operation.name": "execute_tool"}
            if err: a["error.type"] = err
            self.hist.record(time.perf_counter() - t0, a)

def join_report(agent_rows, server_rows):
    tids = {r["trace_id"] for r in agent_rows if r["name"].startswith("invoke_agent")}
    cli = {r["span_id"]: r for r in agent_rows}
    calls = [r for r in server_rows if r["name"].startswith("tools/call")]
    joined = [r for r in calls if r["trace_id"] in tids]
    ov = [((cli[r["parent_span_id"]]["end_unix_nano"] - cli[r["parent_span_id"]]["start_unix_nano"]) - (r["end_unix_nano"] - r["start_unix_nano"])) / 1e6
          for r in joined if r["parent_span_id"] in cli]
    by_tool = {}
    for r in calls: by_tool[r["attributes"].get("gen_ai.tool.name")] = by_tool.get(r["attributes"].get("gen_ai.tool.name"), 0) + 1
    return {"tool_calls": len(calls), "joined": len(joined), "orphans": len(calls) - len(joined),
            "overhead_ms_median": float(np.median(ov)) if ov else None, "by_tool": by_tool}

def safe_call(bridge, name, args, timeout_s, retries=1, backoff_s=0.05):
    read_only = TOOL_META.get(name, {}).get("read_only", False)
    if not read_only and "idempotency_key" not in args:
        args = {**args, "idempotency_key": hashlib.sha256(json.dumps([name, args], sort_keys=True).encode()).hexdigest()[:16]}
    last = None
    for attempt in range(1, retries + 2):
        try:
            return {"ok": True, "value": bridge.call(name, args, timeout_s=timeout_s), "error_type": None, "attempts": attempt}
        except ToolError as e:
            last = e.error_type
            if e.error_type in ("timeout", "connection_closed") and attempt <= retries:
                time.sleep(backoff_s); timeout_s = max(timeout_s, 3.0) if not read_only else timeout_s; continue
            return {"ok": False, "value": None, "error_type": last, "attempts": attempt}
    return {"ok": False, "value": None, "error_type": last, "attempts": retries + 1}

def tools_digest(tools):
    items = sorted([[t.name, t.description or "", getattr(t, "input_schema", None) or {}] for t in tools], key=lambda x: x[0])
    return hashlib.sha256(json.dumps(items, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]

def detect_drift(approved, tools):
    cur = per_tool_digest(tools)
    added, removed = sorted(set(cur) - set(approved)), sorted(set(approved) - set(cur))
    changed = sorted(n for n in set(cur) & set(approved) if cur[n] != approved[n])
    return {"drifted": bool(added or removed or changed), "added": added, "removed": removed, "changed": changed}

def spec_gaps(span_row, kind):
    present = span_row["attributes"]
    return {lvl: sorted(a for a in attrs if a not in present) for lvl, attrs in SPEC[kind].items()}

display(run_rubric())